# Bronze — Yahoo Finance Ingestion

Fetches daily market data from 2022-01-01 to today for ISK exchange rates and the OMX Iceland All-Share Index.

**Source:** Yahoo Finance via `yfinance`  
**Tickers:** `EURUSD=X`, `ISKUSD=X`, `^OMX`  
**Output:** `bronze.yahoo_finance_raw` (Delta table)  
**Schedule:** Daily  

In [ ]:
import yfinance as yf
import pandas as pd
import re
from datetime import timedelta

TICKERS = ["EURUSD=X", "ISKUSD=X", "^OMX"]
INTERVAL = "1d"

if spark.catalog.tableExists("bronze.yahoo_finance_raw"):
    last_date = spark.sql("SELECT MAX(date) FROM bronze.yahoo_finance_raw").collect()[0][0]
    start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"Incremental load — fetching from {start_date}")
else:
    last_date = None
    start_date = "2022-01-01"
    print(f"Full load — fetching from {start_date}")

In [ ]:
today = pd.Timestamp.now().strftime("%Y-%m-%d")

if start_date >= today:
    print(f"Table is already up to date (last date: {last_date}). Nothing to fetch.")
    raw_data = None
else:
    raw_data = yf.download(
        tickers=TICKERS,
        start=start_date,
        interval=INTERVAL,
        auto_adjust=True
    )

    # Flatten MultiIndex columns (price_type, ticker) and sanitise names
    # re.sub collapses multiple underscores (e.g. close__omx -> close_omx from ^OMX ticker)
    raw_data.columns = [
        re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]', '_', f"{price}_{ticker}")).strip('_').lower()
        for price, ticker in raw_data.columns
    ]

    raw_data = raw_data.reset_index()
    raw_data.columns = [
        re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]', '_', c)).strip('_').lower()
        for c in raw_data.columns
    ]

    raw_data['ingested_at'] = pd.Timestamp.now()

    print(f"Fetched {len(raw_data)} rows | Columns: {raw_data.columns.tolist()}")

    if len(raw_data) == 0:
        raise ValueError("[bronze_yahoo_finance] yfinance returned 0 rows - halting.")

In [ ]:
if raw_data is not None:
    spark_df = spark.createDataFrame(raw_data)
    spark_df.write.format("delta").mode("append").saveAsTable("bronze.yahoo_finance_raw")
    print(f"Appended {spark_df.count()} rows to bronze.yahoo_finance_raw")
else:
    print("Nothing to write — table is already up to date.")